# M23 — Implement a Forward Pass with NumPy

**Objective:** reconstruct inference from array operations.

M22 opened one neuron and one dense layer as arithmetic. M23 asks what
a **stack** of those layers is. The useful whole is not a training loop.
It is a named computational graph:

`X → Z1 = X @ W1 + b1 → H = ReLU(Z1) → logits = H @ W2 + b2 → probabilities = softmax(logits)`

with the same row-batch layout as M16/M22: `X` is `(batch, n_in)` and
each `W` is `(n_in, n_out)`.

Gradients, credit assignment, and autograd stay closed (M24-M25).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a number, a sign, a shape, or which named
array should move.

Do not write `backward`, do not import a framework, and do not treat
argmax as a substitute for intermediate parity. If a failure can be
diagnosed from one-example versus batch disagreement or a named
intermediate, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M23" / "forward_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M23.forward_core import (
    BATCH_AXIS,
    CLASS_AXIS,
    DEFAULT_ATOL,
    GRAPH_NODES,
    HIDDEN_ACTIVATION,
    REFERENCE_B1,
    REFERENCE_B2,
    REFERENCE_HIDDEN_ACTIVATION,
    REFERENCE_HIDDEN_PREACTIVATION,
    REFERENCE_LOGITS,
    REFERENCE_PROBABILITIES,
    REFERENCE_W1,
    REFERENCE_W2,
    REFERENCE_X,
    affine,
    arrays_close,
    forward_report,
    intermediate_parity,
    m22_reference_forward,
    perturb_matrix,
    probability_row_sums,
    reorder_rows,
    scalar_two_layer_one_example,
    shift_logits,
    singleton_batch_parity,
    stacked_shapes,
    stable_softmax,
    two_layer_forward,
    two_layer_forward_with_defect,
    validate_stack_shapes,
)
from missions.M22.neuron_layer_core import (
    REFERENCE_LAYER_BIAS,
    REFERENCE_LAYER_W,
    REFERENCE_LAYER_X,
    dense_forward,
)

print("repository root:", ROOT)
print("graph:", GRAPH_NODES)
print("layout: X (batch, n_in) @ W (n_in, n_out) + b (n_out,)")
print("softmax axis:", CLASS_AXIS, "(class axis; batch axis is", BATCH_AXIS, ")")
print("atol:", DEFAULT_ATOL, "hidden activation:", HIDDEN_ACTIVATION)
print("M22 first-layer X matches M23 X:", REFERENCE_LAYER_X == REFERENCE_X)


## M22 boundary: keep the layer convention

M22 froze `Y = activation(X @ W + b)` with `W` shaped `(n_in, n_out)`
and activation **after** the affine map. M23 reuses that contract. It
does not invent a column-vector lecture or a `(n_out, n_in)` weight
layout.

What this mission **opens:** NumPy matmul, bias broadcast, two-layer
composition, named intermediates, batches, logits, and stable softmax.

What stays **deferred:**
- M24 — backpropagation and parameter gradients
- M25 — PyTorch autograd / training loop

Changing one weight to see which outputs move is an inference probe.
It is not credit assignment.


## Frozen teaching fixtures

Declare the useful whole **before** the first calculation.

| Fixture | Value |
| --- | --- |
| `X` | two rows, three features (M22 `REFERENCE_LAYER_X`) |
| `W1` | `(3, 2)` so `n_in=3`, `n_hidden=2` |
| `b1` | `(0.0, 0.5)` |
| Hidden map | ReLU after the first affine map |
| `W2` | `(2, 3)` so three class logits |
| `b2` | `(0.0, 0.0, 0.0)` |
| Softmax | class axis = last axis |
| Dtype / tolerance | `float64`, `atol=1e-12`, `rtol=0` (proposed default; ADR unfilled) |

Named intermediates, in order: `hidden_preactivation`,
`hidden_activation`, `logits`, `probabilities`.

Primary sources: `numpy-quickstart` and `3b1b-neural-networks` in
`data/source_registry.json`. Skip micrograd and PyTorch here.


## Predict before running — first layer from M22

Timestamp a prediction before `run-first-layer`.

`X` is `(2, 3)`, `W1` is `(3, 2)`, `b1` is `(2,)`. Predict:
- first-row pre-activation is `(0.0, -0.5)` and ReLU hides it to `(0, 0)`
- second-row pre-activation is `(1.0, 1.5)` and ReLU leaves it
- `dense_forward` from M22 and `affine` plus ReLU in M23 agree

Activation happens **after** `X @ W1 + b1`.


In [ ]:
print("validate_stack_shapes", validate_stack_shapes(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2))
print("stacked_shapes", stacked_shapes(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2))
z1 = affine(REFERENCE_X, REFERENCE_W1, REFERENCE_B1)
m22_hidden = dense_forward(REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, "relu")
m22_pre = dense_forward(REFERENCE_LAYER_X, REFERENCE_LAYER_W, REFERENCE_LAYER_BIAS, "identity")
print("M23 affine Z1", z1)
print("M22 identity first layer", m22_pre)
print("M22 ReLU first layer", m22_hidden)
assert arrays_close(z1, REFERENCE_HIDDEN_PREACTIVATION)
assert arrays_close(m22_hidden, REFERENCE_HIDDEN_ACTIVATION)
assert arrays_close(m22_pre, z1)
print("first-layer M22/M23 parity holds")


### The first layer is still M22

Row 0 is the ReLU hinge: a negative unit becomes 0. Row 1 is entirely
non-negative, so ReLU is a copy. That is inherited evidence, not a new
layer convention. M23's job is to stack the next map without losing
these names.


## Predict before running — scalar-to-vectorized parity

Timestamp a prediction before `run-scalar`.

Replace the batched matmul with explicit per-example loops on **the
same** `X`, `W1`, `b1`, `W2`, `b2`, and ReLU.

Predict that every named intermediate of row 0 and row 1 matches the
later vectorized stack within `atol=1e-12`.


In [ ]:
scalar_rows = [
    scalar_two_layer_one_example(row, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
    for row in REFERENCE_X
]
for index, scalar in enumerate(scalar_rows):
    print("row", index)
    print("  hidden_preactivation", scalar.hidden_preactivation)
    print("  hidden_activation", scalar.hidden_activation)
    print("  logits", scalar.logits)
    print("  probabilities", scalar.probabilities)
assert arrays_close(scalar_rows[0].hidden_activation, (0.0, 0.0))
assert arrays_close(scalar_rows[1].logits, REFERENCE_LOGITS[1])
print("scalar path produced the hand-computed graph")


### Loops make the axes obvious

The scalar path has no batch axis to confuse with the class axis. If a
later vectorized run disagrees with these tuples, the bug is in
broadcasting or in which axis softmax reduced — not in "neural network
mystery."


## Predict before running — vectorize the batch

Timestamp a prediction before `run-vectorized-batch`.

Predict:
- `two_layer_forward` on the full `X` reproduces each scalar row
- a singleton run on one row has shape `(1, 3)` for logits and probabilities
- bias `(n_out,)` broadcasts along the batch because the trailing axis matches


In [ ]:
batch = two_layer_forward(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
print("batch shapes", batch.shapes)
print(forward_report(batch))
for index, scalar in enumerate(scalar_rows):
    single = two_layer_forward(REFERENCE_X[index], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
    print("row", index, "singleton logits", single.logits, "batch logits", batch.logits[index])
    assert single.logits.shape == (1, 3)
    assert arrays_close(single.probabilities[0], batch.probabilities[index])
    assert arrays_close(scalar.probabilities, batch.probabilities[index])
parity = singleton_batch_parity(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
print("singleton_batch_parity", parity)
assert parity["all_match"]
print("vectorized batch matches the scalar rows")


### A batch is stacked singletons

`X @ W + b` uses the NumPy trailing-axis broadcast from
`numpy-quickstart`: `b` has shape `(n_out,)`, so it is added to every
row. If row `i` of the batch disagrees with the singleton run on that
row, either the layout drifted from M16 or softmax reduced the wrong
axis.


## Predict before running — compose the next layer

Timestamp a prediction before `run-compose`.

The second affine map reads the **hidden activation**, not the raw
features. Predict the named graph:

- `hidden_preactivation` row 0 = `(0.0, -0.5)`
- `hidden_activation` row 0 = `(0.0, 0.0)`
- `logits` row 0 = `(0.0, 0.0, 0.0)`
- `logits` row 1 = `(1.0, 1.5, -0.25)`

`GRAPH_NODES` is the computational graph M24 will later walk backward.
Do not walk it backward here.


In [ ]:
print("GRAPH_NODES", GRAPH_NODES)
print("hidden_preactivation", batch.hidden_preactivation)
print("hidden_activation", batch.hidden_activation)
print("logits", batch.logits)
assert arrays_close(batch.hidden_preactivation, REFERENCE_HIDDEN_PREACTIVATION)
assert arrays_close(batch.hidden_activation, REFERENCE_HIDDEN_ACTIVATION)
assert arrays_close(batch.logits, REFERENCE_LOGITS)
print("named intermediates match the hand-computed graph")


### Names localize mismatches

If only probabilities look "off," check logits. If logits are off but
the hidden activation matches M22, the second layer is the suspect. If
the hidden activation is off, stop before softmax. Final argmax is a
lossy summary of this graph, not a diagnosis.


## Predict before running — logits versus stable softmax

Timestamp a prediction before `run-softmax`.

Logits are unnormalized class scores. Softmax along the **class axis**
turns one row into a probability simplex.

Predict:
- row 0 is uniform, so each class is `1/3`
- row 1 puts the most mass on class 1 because `1.5` is the largest logit
- probabilities are non-negative and each row sums to 1
- `CLASS_AXIS` is `-1`, not the batch axis


In [ ]:
probs = stable_softmax(batch.logits, axis=CLASS_AXIS)
print("CLASS_AXIS", CLASS_AXIS)
print("probabilities", probs)
print("row sums", probability_row_sums(probs))
print("independent softmax on logits", stable_softmax(REFERENCE_LOGITS))
assert arrays_close(probs, batch.probabilities)
assert arrays_close(probs, REFERENCE_PROBABILITIES)
assert arrays_close(probability_row_sums(probs), (1.0, 1.0))
assert abs(float(probs[0, 0]) - (1.0 / 3.0)) < DEFAULT_ATOL
assert int(probs[1].argmax()) == 1
print("class-axis softmax produced a simplex")


### Logits are not probabilities

Row 0's logits are all zero, so the classes are tied. Softmax does not
invent a winner; it reports the tie as `1/3` each. Row 1's logits are
ordered `(1.0, 1.5, -0.25)`, so class 1 is preferred. The map is
`exp` then normalize. Subtracting the row maximum first is a stability
transform, not a change of relative scores.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
indices = range(3)
width = 0.35
ax.bar([i - width / 2 for i in indices], list(batch.probabilities[0]), width=width, label="example 0")
ax.bar([i + width / 2 for i in indices], list(batch.probabilities[1]), width=width, label="example 1")
ax.set(xlabel="class", ylabel="probability", title="Softmax along the class axis", ylim=(0, 1))
ax.legend()
fig.tight_layout()
plt.show()


## Predict before running — softmax shift invariance

Timestamp a prediction before `run-shift`.

Add the same constant `5.0` to **all three logits of example 1 only**.
Relative logits and class order stay fixed.

Predict that example 1's probabilities are unchanged within `atol=1e-12`,
and that example 0 is untouched.


In [ ]:
shifted_logits = shift_logits(batch.logits, example_index=1, constant=5.0)
shifted_probs = stable_softmax(shifted_logits, axis=CLASS_AXIS)
print("original logits row 1", batch.logits[1])
print("shifted logits row 1", shifted_logits[1])
print("original probs row 1", batch.probabilities[1])
print("shifted probs row 1", shifted_probs[1])
assert arrays_close(shifted_probs[0], batch.probabilities[0])
assert arrays_close(shifted_probs[1], batch.probabilities[1])
large = stable_softmax((1000.0, 1001.0, 999.0))
print("large-logit softmax", large)
assert abs(float(sum(large)) - 1.0) < DEFAULT_ATOL
print("shift invariance holds; large logits stay finite")


### A constant shift is not a new prediction

Softmax sees differences. Adding 5 to every class of one example is
the same comparison. The max-subtraction used for stability is this
fact put to work on large numbers. It is still not a gradient.


## Predict before running — reference parity

Timestamp a prediction before `run-reference`.

Run the trusted M22 path: `dense_forward` for the hidden layer, then
`dense_forward` with identity for logits, then M23 class-axis softmax.

Predict that **every** named intermediate matches `two_layer_forward`
within the declared dtype/tolerance. If only the probabilities match,
that is not enough for M24.


In [ ]:
trusted = m22_reference_forward(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
parity = intermediate_parity(batch, trusted)
print("intermediate_parity", parity)
for name, ok in parity.items():
    print(name, "match" if ok else "MISMATCH")
assert all(parity.values())
assert arrays_close(trusted.hidden_activation, dense_forward(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, "relu"))
print("NumPy stack matches the M22 composition on every named intermediate")


### Trusted forward-parity is the M24 handoff

M24 may start from this graph only if hidden pre-activation, hidden
activation, logits, and probabilities all agree. An argmax check would
hide a wrong-axis softmax that still picked the same winner.


## Predict before running — single-weight perturbation

Timestamp a prediction before `run-perturbation`.

Change **exactly one** entry: `W2[0, 0] += 0.1` (hidden unit 0 → class 0).
No retraining. All other parameters stay fixed.

Predict:
- hidden intermediates do not move (`W2` is after the hidden layer)
- example 0 logits stay put because its hidden unit 0 is 0
- example 1 class-0 logit rises by `0.1 * 1.0 = 0.1`
- example 1 probabilities move; example 0 probabilities do not


In [ ]:
weights_perturbed = perturb_matrix(REFERENCE_W2, (0, 0), 0.1)
changed = two_layer_forward(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, weights_perturbed, REFERENCE_B2)
print("hidden unchanged", arrays_close(batch.hidden_activation, changed.hidden_activation))
print("example 0 logits", batch.logits[0], "->", changed.logits[0])
print("example 1 logits", batch.logits[1], "->", changed.logits[1])
print("delta example 1 class 0", float(changed.logits[1, 0] - batch.logits[1, 0]))
assert arrays_close(batch.hidden_activation, changed.hidden_activation)
assert arrays_close(batch.logits[0], changed.logits[0])
assert abs(float(changed.logits[1, 0] - batch.logits[1, 0]) - 0.1) < DEFAULT_ATOL
assert arrays_close(batch.probabilities[0], changed.probabilities[0])
assert not arrays_close(batch.probabilities[1], changed.probabilities[1])
print("perturbation stayed on the intended node")


### One weight, one local effect

This is still inference. We did not compute a derivative, did not take
a step, and did not mention a loss. The graph tells us which arrays
are downstream of `W2[0, 0]`. That localization is what M24 will later
turn into blame; here it is only a forward probe.


## Predict before running — batch reordering

Timestamp a prediction before `run-reorder`.

Reorder input rows to `(1, 0)`. Parameters and feature order stay fixed.

Predict that outputs reorder the same way and that restoring the row
order restores the original probabilities. A mixed-up row would mean
cross-example contamination.


In [ ]:
shuffled_x = reorder_rows(REFERENCE_X, (1, 0))
shuffled = two_layer_forward(shuffled_x, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
print("original probs", batch.probabilities)
print("reordered probs", shuffled.probabilities)
assert arrays_close(shuffled.probabilities[0], batch.probabilities[1])
assert arrays_close(shuffled.probabilities[1], batch.probabilities[0])
assert arrays_close(reorder_rows(shuffled.probabilities, (1, 0)), batch.probabilities)
print("reordering permutes rows without mixing them")


### Rows are examples, not a class axis

If softmax had reduced across the batch, swapping examples would not
be a pure permutation of output rows. Keep that observation; the
controlled failure uses it.


## Code reading — validate, matmul, activate, stabilize

Read `two_layer_forward` and `stable_softmax` in
`missions/M23/forward_core.py` (see also `missions/M23/code_reading.md`).
Before the next cell, predict:

1. whether a 1-D `x` becomes a batch of one row
2. what error you get if `b1` has the wrong length
3. which axis `stable_softmax` reduces by default
4. why `keepdims=True` on the max/sum is required for broadcasting

Do not search the file for a backward pass.


In [ ]:
forward_src = inspect.getsource(two_layer_forward)
softmax_src = inspect.getsource(stable_softmax)
for marker in ("validate_stack_shapes", "affine", "apply_hidden_activation", "stable_softmax"):
    print(f"two_layer_forward contains {marker!r}: {marker in forward_src}")
print("stable_softmax default axis is CLASS_AXIS:", "CLASS_AXIS" in softmax_src)
print("stability uses keepdims:", "keepdims=True" in softmax_src)
print("GRAPH_NODES", GRAPH_NODES)
try:
    validate_stack_shapes(REFERENCE_X, REFERENCE_W1, (0.0,), REFERENCE_W2, REFERENCE_B2)
    raise AssertionError("expected a bias length mismatch")
except ValueError as exc:
    print("malformed bias:", exc)
one = two_layer_forward(REFERENCE_X[0], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)
print("1-D x becomes", one.x.shape)


## Predict before running — Controlled failure: softmax across the batch axis

Timestamp a prediction before `run-failure`.

The intended softmax reduces the class axis. The defective path uses
`defect="softmax_axis_batch"` and reduces axis 0 instead.

Predict:
- a singleton run on example 0 emits `(1, 1, 1)` because each class is
  a one-element softmax
- the batch run no longer matches that singleton row
- batch **columns** sum to 1; rows generally do not
- logits are still the hand-computed values — only the last node is wrong


In [ ]:
broken_axis = two_layer_forward_with_defect(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="softmax_axis_batch",
)
single_broken = two_layer_forward_with_defect(
    REFERENCE_X[0], REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="softmax_axis_batch",
)
axis_parity = singleton_batch_parity(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="softmax_axis_batch",
)
print("broken probabilities", broken_axis.probabilities)
print("singleton broken probabilities", single_broken.probabilities)
print("singleton_batch_parity", axis_parity)
print("logits still trusted", arrays_close(broken_axis.logits, REFERENCE_LOGITS))
assert arrays_close(single_broken.probabilities[0], (1.0, 1.0, 1.0))
assert not axis_parity["all_match"]
assert arrays_close(broken_axis.logits, batch.logits)
print("wrong-axis softmax is visible in one-example versus batch parity")


### Diagnose before repair

Symptom: probabilities look numeric, maybe even "like probabilities,"
but a singleton disagrees with its batch row.

Plausible hypotheses include a wrong softmax axis, an omitted hidden
activation, or a transposed weight. The discriminating experiment is
already on the table: one-example versus batch parity plus a logit
check. Logits still match, so the affine stack is innocent. The last
node reduced the batch axis.

Do not repair this by changing weights or by opening M24.


## Predict before running — Controlled failure: omitted hidden activation

Timestamp a prediction before `run-omitted`.

Keep class-axis softmax. Skip ReLU (`defect="omitted_hidden_activation"`).

Predict:
- singleton versus batch **still matches** (the bug is not an axis)
- example 0 hidden activation is `(0.0, -0.5)` instead of `(0.0, 0.0)`
- example 1 is unchanged because both pre-activations were positive
- intermediate mismatch localizes the defect before probabilities


In [ ]:
omitted = two_layer_forward_with_defect(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    hidden_activation="relu",
    defect="omitted_hidden_activation",
)
omitted_parity = singleton_batch_parity(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="omitted_hidden_activation",
)
print("omitted hidden activation", omitted.hidden_activation)
print("omitted logits", omitted.logits)
print("singleton_batch_parity", omitted_parity)
print("intermediate_parity versus correct", intermediate_parity(batch, omitted))
assert omitted_parity["all_match"]
assert arrays_close(omitted.hidden_activation[0], (0.0, -0.5))
assert arrays_close(omitted.hidden_activation[1], batch.hidden_activation[1])
assert not arrays_close(omitted.logits[0], batch.logits[0])
print("omitted ReLU survives row-parity and fails the named-intermediate check")


### Two defects, two discriminators

Wrong-axis softmax breaks one-example versus batch parity. Omitted
ReLU does not; it shows up on `hidden_activation` for the mixed-sign
row. Diagnosis uses the graph, not a vibe that "the probabilities look
off." Smallest repair restores one named boundary at a time.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `defect="none"` restores the hand-computed probabilities
exactly, with the same `X`, weights, and biases. Do not change the
teaching activation and do not add a loss.


In [ ]:
repaired = two_layer_forward_with_defect(
    REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2,
    defect="none",
)
print("repaired probabilities", repaired.probabilities)
print("matches batch", arrays_close(repaired.probabilities, batch.probabilities))
assert arrays_close(repaired.probabilities, batch.probabilities)
assert arrays_close(repaired.hidden_activation, REFERENCE_HIDDEN_ACTIVATION)
print("class-axis softmax and hidden ReLU restored the graph")


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- the hand-computed first layer and logits
- scalar versus vectorized parity
- annotated named intermediates and shapes
- softmax shift invariance
- M22 reference parity on every named array
- wrong-axis diagnosis (symptom, hypotheses, discriminator, repair)
- the code-reading trace

See `missions/M23/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M23/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M23/adr_prompt.md` to choose a V05 **inference-parity and
numerical-tolerance** policy. Compare named intermediates, not only
argmax. Do not claim global optimality.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR.


## M22 → M23 → M24 handoff

M22 opened the neuron and one dense layer. M23 reconstructed a
multi-layer NumPy forward pass with named intermediates and a trusted
forward-parity test against `dense_forward`.

M24 may assign blame with backpropagation **only after** these graph
nodes, class-axis softmax, and parity checks are defended.

M24 still owns gradients. M25 still owns autograd.


## Mission summary prompt

In your own words, using only numbers from this lab:

1. Why are logits not probabilities?
2. Why did a singleton `(1, 1, 1)` diagnose batch-axis softmax?
3. Why did omitting ReLU not break row-parity but still fail the graph?
4. What must M24 receive that M22 could not provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert arrays_close(batch.hidden_activation, REFERENCE_HIDDEN_ACTIVATION)
assert arrays_close(batch.logits, REFERENCE_LOGITS)
assert arrays_close(batch.probabilities, REFERENCE_PROBABILITIES)
assert all(intermediate_parity(batch, trusted).values())
assert singleton_batch_parity(REFERENCE_X, REFERENCE_W1, REFERENCE_B1, REFERENCE_W2, REFERENCE_B2)["all_match"]
assert not axis_parity["all_match"]
assert arrays_close(repaired.probabilities, batch.probabilities)
print("M23 integrity checks passed")
